In [ ]:
import os
import json
import cv2
from ultralytics import YOLO

def analizar_y_guardar_json(ruta_imagen):
    modelo = YOLO("best_espanol.pt")
    resultados = modelo.predict(source=ruta_imagen, save=False, conf=0.25)
    resultado = resultados[0]
    nombre_base = os.path.splitext(os.path.basename(ruta_imagen))[0]
    reporte_json = {
        "total_detecciones": len(resultado.boxes),
        "detecciones": []
    }
    
    print("\n=========================================")
    print("      REPORTE DE DAÑOS DETECTADOS        ")
    print("=========================================")
    
    if len(resultado.boxes) == 0:
        print("No se detectó ningún daño en el vehículo.")
    else:
        for box in resultado.boxes:
            id_clase = int(box.cls)
            nombre_dano = modelo.names[id_clase]
            confianza = float(box.conf) * 100
            print(f"-> {nombre_dano.upper()} (Confianza: {confianza:.2f}%)")
            reporte_json["detecciones"].append({
                "descripcion": nombre_dano,
                "porcentaje_confianza": round(confianza, 2)
            })
            
    print("=========================================\n")

    nombre_json = f"reporte_{nombre_base}.json"
    with open(nombre_json, "w", encoding="utf-8") as archivo_json:
        json.dump(reporte_json, archivo_json, indent=4, ensure_ascii=False)
    print(f"¡Reporte JSON guardado como: '{nombre_json}'!")
    
    nombre_imagen_salida = f"resultado_{nombre_base}.jpg"
    imagen_dibujada = resultado.plot()
    cv2.imwrite(nombre_imagen_salida, imagen_dibujada)
    print(f"¡Imagen etiquetada guardada como: '{nombre_imagen_salida}'!")


if __name__ == "__main__":
    analizar_y_guardar_json("auto_chocado4.jpg")